# Pakistan's Islamic Economic Transition: An Empirical and Analytical Framework

## Beyond Prohibition: Building an Economy That Delivers Value

This analysis addresses a fundamental gap in Islamic economics literature: the absence of a
"doable, seasoned concept for how to build an economy that delivers value, governed by sound
institutions" (a common critique of the field). Rather than repeating that interest is prohibited,
we ask: **What specific mechanisms, institutions, and policies would Pakistan need to transition
to a genuinely Islamic economic system — and where do honest gaps in knowledge remain?**

### Structure

1. **Current State Assessment** — Where Pakistan stands empirically
2. **The Policy Rate Question** — What replaces the SBP policy rate?
3. **Government Financing** — What replaces T-bills and PIBs?
4. **Institutional Design** — What new institutions are needed?
5. **Transition Modeling** — DSGE simulations of phased transition
6. **Comparative Lessons** — Iran (full conversion), Malaysia (dual system)
7. **Honest Assessment** — What we don't know yet

### Methodology

- Empirical data from State Bank of Pakistan Islamic Banking Bulletins
- IMF Article IV consultation reports
- Extended Pakistan DSGE model with transition scenarios
- Comparative institutional analysis

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import sys
import os

# Add utilities to path
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'Code'))
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd())))

try:
    from utils.dsge_utilities import (
        BaseIslamicNKModel, ConventionalNKModel,
        compute_impulse_responses, simulate_business_cycle,
        plot_irf_panel, plot_business_cycles,
        COLORS, STYLE, apply_style
    )
except ImportError:
    sys.path.insert(0, '../')
    from Code.utils.dsge_utilities import (
        BaseIslamicNKModel, ConventionalNKModel,
        compute_impulse_responses, simulate_business_cycle,
        plot_irf_panel, plot_business_cycles,
        COLORS, STYLE, apply_style
    )

apply_style()

# Figure output
fig_dir = os.path.join(os.path.dirname(os.getcwd()), 'Website', 'theory')
os.makedirs(fig_dir, exist_ok=True)
print(f'Figure output: {fig_dir}')

---
## Part I: Current State Assessment

### 1.1 Pakistan's Banking Sector: Conventional vs Islamic

Pakistan operates a **dual banking system** where Islamic banks compete alongside
conventional banks under the State Bank of Pakistan's regulatory framework.

In [ ]:
# Pakistan Islamic Banking Growth (SBP Islamic Banking Bulletin data)
# Source: State Bank of Pakistan, Islamic Banking Department

islamic_banking_data = pd.DataFrame({
    'Year': [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025],
    'IB_Assets_Share_Pct': [11.4, 11.7, 12.4, 13.5, 15.2, 16.0, 17.5, 19.5, 20.0, 20.6, 21.3],
    'IB_Deposits_Share_Pct': [13.2, 13.7, 14.5, 15.6, 17.0, 17.8, 19.1, 20.8, 21.5, 22.1, 22.8],
    'IB_Branches': [1860, 2044, 2320, 2581, 2851, 2949, 3184, 3490, 3716, 3890, 4050],
    'Full_Islamic_Banks': [5, 5, 5, 5, 5, 5, 5, 5, 6, 6, 6],
    'Conv_With_IB_Windows': [16, 16, 16, 16, 17, 17, 17, 17, 17, 17, 17],
    'IB_NPL_Ratio_Pct': [4.8, 4.5, 4.1, 4.3, 4.7, 5.0, 3.9, 3.5, 3.2, 3.0, 2.8],
    'Conv_NPL_Ratio_Pct': [11.4, 10.1, 8.4, 7.9, 8.6, 9.2, 7.6, 7.3, 7.1, 6.8, 6.5],
})

# Plot: Islamic Banking Market Share Growth
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel 1: Asset share
axes[0].bar(islamic_banking_data['Year'], islamic_banking_data['IB_Assets_Share_Pct'],
            color=COLORS['primary'], alpha=0.8)
axes[0].set_title('Islamic Banking: Share of Total Assets', fontweight='bold')
axes[0].set_ylabel('Percentage (%)')
axes[0].set_ylim(0, 30)
for i, v in enumerate(islamic_banking_data['IB_Assets_Share_Pct']):
    axes[0].text(islamic_banking_data['Year'].iloc[i], v + 0.5, f'{v}%', 
                ha='center', fontsize=8)

# Panel 2: Branch network
axes[1].plot(islamic_banking_data['Year'], islamic_banking_data['IB_Branches'],
             color=COLORS['primary'], marker='o', linewidth=2)
axes[1].fill_between(islamic_banking_data['Year'], islamic_banking_data['IB_Branches'],
                     alpha=0.1, color=COLORS['primary'])
axes[1].set_title('Islamic Banking Branch Network', fontweight='bold')
axes[1].set_ylabel('Number of Branches')

# Panel 3: NPL comparison
x = islamic_banking_data['Year']
width = 0.35
axes[2].bar(x - width/2, islamic_banking_data['IB_NPL_Ratio_Pct'], width,
            label='Islamic Banks', color=COLORS['primary'], alpha=0.8)
axes[2].bar(x + width/2, islamic_banking_data['Conv_NPL_Ratio_Pct'], width,
            label='Conventional Banks', color=COLORS['orange'], alpha=0.8)
axes[2].set_title('Non-Performing Loans: Islamic vs Conventional', fontweight='bold')
axes[2].set_ylabel('NPL Ratio (%)')
axes[2].legend()

plt.tight_layout()
plt.savefig(os.path.join(fig_dir, 'fig_pakistan_ib_overview.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: fig_pakistan_ib_overview.png')

### 1.2 Key Finding: Islamic Banks Outperform on Asset Quality

Islamic banks in Pakistan consistently show **lower NPL ratios** than conventional banks.
This is not merely cosmetic. The profit-and-loss sharing structure creates stronger
incentives for due diligence on the bank's part, since the bank shares in actual business
risk rather than simply lending at interest.

However, a critical question remains: **how much of Pakistan's Islamic banking is genuinely
different from conventional banking, and how much is repackaged interest under Shariah-
compliant labels?**

### 1.3 The Honesty Gap: Product Composition Analysis

Pakistan's Islamic banking products break down roughly as follows:

| Product | Share of IB Financing | Genuinely Different? | Notes |
|---------|----------------------|---------------------|-------|
| **Diminishing Musharakah** | ~35% | Partially | Home/auto finance; structured like mortgages but with declining co-ownership |
| **Murabaha** (cost-plus) | ~25% | Minimally | Fixed markup functionally resembles interest; pre-determined return |
| **Ijarah** (leasing) | ~18% | Partially | Asset-backed leasing; genuinely ties return to asset use |
| **Salam/Istisna** | ~8% | Yes | Forward sale/manufacturing contracts; genuine trade finance |
| **Musharakah/Mudarabah** | ~7% | Yes | True profit-loss sharing; highest risk for the bank |
| **Other** | ~7% | Mixed | Various instruments |

**Critical observation:** Only about 15% of Islamic banking financing in Pakistan involves
genuine profit-loss sharing (Musharakah/Mudarabah). The bulk uses fixed-return structures
(Murabaha, Diminishing Musharakah) that, while legally distinct, produce economic outcomes
similar to conventional interest-based lending.

This is the honest gap the critic identifies. The system has Shariah-compliant *form* but
often lacks the risk-sharing *substance* that Islamic economics theoretically requires.

---
## Part II: The Central Question — What Replaces the Policy Rate?

### 2.1 How the SBP Currently Conducts Monetary Policy

The State Bank of Pakistan uses a conventional **interest rate corridor** system:

- **Policy Rate** (currently ~12%): The target overnight interbank rate
- **Ceiling**: SBP Reverse Repo Rate (Policy Rate + 100 bps)
- **Floor**: SBP Repo Rate (Policy Rate - 100 bps)
- **Instruments**: Open Market Operations (OMOs) via T-bill repos
- **Islamic OMOs**: Shariah-compliant Mudarabah-based OMOs run in parallel

The fundamental problem: **Islamic banks participate in a system whose benchmark is an
interest rate.** Even Mudarabah-based OMOs effectively mirror the conventional policy rate.
The SBP sets a single monetary policy stance that both systems follow.

### 2.2 Three Proposed Alternatives to the Interest-Based Policy Rate

The academic literature proposes several alternatives. We analyze each for feasibility:

In [ ]:
# Comparative analysis of monetary policy alternatives

alternatives = pd.DataFrame({
    'Mechanism': [
        'Profit-Rate Targeting',
        'Reserve Requirement Ratio',
        'Credit Allocation Quotas',
        'Asset-Based Benchmark',
        'Dual Rate System (Transition)',
    ],
    'Description': [
        'Central bank targets an economy-wide profit rate derived from real sector returns',
        'Adjust required reserves to control money supply directly (quantity-based)',
        'Direct lending quotas by sector, prioritizing productive over speculative use',
        'Benchmark tied to real asset returns (e.g., weighted average rental yields)',
        'Maintain conventional rate as benchmark while building parallel Islamic rate',
    ],
    'Precedent': [
        'Theoretical only; no country has implemented at scale',
        'China uses RRR extensively; pre-1980s Western central banks',
        'South Korea (1960s-80s), India (priority sector lending)',
        'No precedent; conceptual',
        'Malaysia (de facto); Pakistan (current partial approach)',
    ],
    'Feasibility': ['Low', 'Medium-High', 'Medium', 'Low', 'High'],
    'Risk': [
        'No reliable real-time profit data; gaming/manipulation',
        'Blunt instrument; may cause credit crunches',
        'Rent-seeking, corruption, misallocation',
        'Data availability; measurement challenges',
        'May perpetuate status quo indefinitely',
    ],
})

print('='*90)
print('MONETARY POLICY ALTERNATIVES: FEASIBILITY ASSESSMENT')
print('='*90)
for _, row in alternatives.iterrows():
    print(f"\n{row['Mechanism']} [Feasibility: {row['Feasibility']}]")
    print(f"  Description: {row['Description']}")
    print(f"  Precedent:   {row['Precedent']}")
    print(f"  Risk:        {row['Risk']}")
print('\n' + '='*90)

### 2.3 The Honest Answer

**No country has successfully operated monetary policy at scale without an interest rate
or its functional equivalent.** Iran renamed its rates but the economic mechanism remained
identical. Malaysia maintains a dual system where the Islamic rate shadows the conventional one.

The most feasible near-term path for Pakistan is a **Dual Rate System** that:

1. Maintains the conventional policy rate as the primary monetary tool (pragmatism)
2. Develops a parallel **Islamic Benchmark Rate** derived from actual mudarabah returns
3. Gradually increases the weight of Islamic instruments in OMOs
4. Builds the institutional infrastructure (Islamic interbank market, benchmark methodology)
5. Sets a 10-15 year timeline for convergence

This is not the ideologically pure answer. But it is the doable one — and the critic asked
for doable.

---
## Part III: Government Financing Without Interest-Based Debt

### 3.1 Pakistan's Current Government Debt Composition

Pakistan finances its fiscal deficit through:
- **Market Treasury Bills (MTBs)**: Short-term (3, 6, 12-month) interest-bearing
- **Pakistan Investment Bonds (PIBs)**: Long-term (3, 5, 10, 15, 20-year) interest-bearing
- **Sukuk**: Shariah-compliant sovereign instruments (~14.5% of domestic securities by Dec 2025)
- **External borrowing**: IMF, bilateral, commercial (interest-based)

Pakistan issued over Rs2 trillion (~$7 billion) in sukuk during 2025, the highest ever.
The government targets 20% Shariah-compliant debt by FY 2027-28.

In [ ]:
# Government debt composition and sukuk trajectory

debt_composition = pd.DataFrame({
    'Year': [2020, 2021, 2022, 2023, 2024, 2025],
    'Sukuk_Share_Pct': [5.2, 7.1, 9.3, 11.0, 12.6, 14.5],
    'Conventional_Share_Pct': [94.8, 92.9, 90.7, 89.0, 87.4, 85.5],
    'Total_Domestic_Debt_TrPKR': [23.2, 26.3, 31.5, 38.7, 43.1, 48.2],
})

# Projection: path to 20% and beyond
projection = pd.DataFrame({
    'Year': [2026, 2027, 2028, 2029, 2030, 2032, 2035],
    'Sukuk_Share_Pct': [16.5, 18.5, 20.0, 23.0, 26.0, 35.0, 50.0],
    'Scenario': ['Target', 'Target', 'Target', 'Ambitious', 'Ambitious', 'Ambitious', 'Ambitious'],
})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Current composition
years = debt_composition['Year']
axes[0].bar(years, debt_composition['Sukuk_Share_Pct'], label='Sukuk (Islamic)',
            color=COLORS['primary'], alpha=0.8)
axes[0].bar(years, debt_composition['Conventional_Share_Pct'],
            bottom=debt_composition['Sukuk_Share_Pct'],
            label='Conventional (Interest-based)', color=COLORS['gray'], alpha=0.5)
axes[0].set_title('Government Domestic Debt Composition', fontweight='bold')
axes[0].set_ylabel('Share (%)')
axes[0].legend(loc='upper left')
axes[0].set_ylim(0, 105)

# Panel 2: Sukuk growth trajectory with projection
actual_years = debt_composition['Year'].tolist()
actual_pct = debt_composition['Sukuk_Share_Pct'].tolist()
proj_years = projection['Year'].tolist()
proj_pct = projection['Sukuk_Share_Pct'].tolist()

axes[1].plot(actual_years, actual_pct, 'o-', color=COLORS['primary'], 
             linewidth=2, markersize=6, label='Actual')
axes[1].plot(proj_years, proj_pct, 's--', color=COLORS['accent'],
             linewidth=2, markersize=6, label='Projected')
axes[1].axhline(y=20, color=COLORS['red'], linestyle=':', alpha=0.5, label='20% Target (FY28)')
axes[1].axhline(y=50, color=COLORS['blue'], linestyle=':', alpha=0.5, label='50% Milestone')
axes[1].fill_between(proj_years, proj_pct, alpha=0.1, color=COLORS['accent'])
axes[1].set_title('Sukuk Share Trajectory: Actual & Projected', fontweight='bold')
axes[1].set_ylabel('Sukuk as % of Domestic Securities')
axes[1].legend()
axes[1].set_ylim(0, 60)

plt.tight_layout()
plt.savefig(os.path.join(fig_dir, 'fig_pakistan_debt_composition.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: fig_pakistan_debt_composition.png')

### 3.2 The Sukuk Problem: Are They Genuinely Different?

Pakistan's sovereign sukuk are primarily **Ijarah-based** (sale-and-leaseback of government
assets). The government sells an asset (e.g., motorway section) to a Special Purpose Vehicle,
then leases it back with periodic rental payments to sukuk holders.

**What's genuinely different:**
- Asset-backed: there is a real asset underlying the security
- Rental payments tied to asset use rather than pure time-value of money
- Default recovery is asset-based, not just a general government obligation

**What's functionally similar:**
- Rental rates are benchmarked to the SBP policy rate (so they move with interest rates)
- Fixed periodic payments resemble coupon payments on bonds
- Secondary market trading and pricing mirrors conventional bonds
- Investors treat them as near-substitutes for PIBs

**The key question for transition:** Can sukuk scale to replace ALL government borrowing
(~Rs48 trillion) without becoming a pure relabeling exercise? This requires:
1. Sufficient government assets to back the sukuk
2. A benchmark rate independent of the conventional policy rate
3. Genuine risk-sharing elements (currently absent from sovereign sukuk)

---
## Part IV: DSGE Transition Scenarios

We extend the Pakistan DSGE model to simulate three transition scenarios:

1. **Gradual (15-year)**: Incremental shift, dual system maintained throughout
2. **Accelerated (7-year)**: Aggressive conversion per Supreme Court ruling
3. **Shock (3-year)**: Forced rapid conversion (Iran 1983 approach)

In [ ]:
class PakistanTransitionModel(BaseIslamicNKModel):
    """
    Pakistan DSGE model extended with transition dynamics.
    
    Key addition: a transition parameter (alpha_t) that represents the share
    of the economy operating under Islamic mechanisms. As alpha_t increases
    from 0 to 1, the monetary transmission mechanism shifts from interest-rate
    based to profit-rate based.
    
    During transition:
    - Monetary rule blends conventional (interest rate) and Islamic (profit rate)
    - Financial intermediation costs may increase (learning/adjustment)
    - Uncertainty premium rises then falls (institutional development curve)
    """
    
    def __init__(self, params=None):
        super().__init__(params)
        
        pakistan_transition_params = {
            # Core Pakistan parameters
            'beta': 0.99,
            'sigma': 1.0,
            'kappa': 0.06,
            'phi_pi': 1.2,
            'phi_y': 0.2,
            'pi_bar': 0.015,
            
            # Fiscal
            'tau_tax': 0.135,
            'debt_gdp': 0.715,
            
            # Transition parameters
            'alpha_0': 0.20,         # Initial Islamic share (current ~20%)
            'alpha_target': 1.0,      # Target Islamic share
            'transition_cost': 0.005, # GDP cost of transition friction per period
            'uncertainty_peak': 0.02, # Peak uncertainty premium during transition
            'learning_rate': 0.05,    # How fast institutions learn (per period)
            
            # Islamic mechanisms (as they replace conventional)
            'profit_rate_spread': 0.002,  # Profit rate tends to be slightly higher
            'pls_dampening': 0.15,        # PLS dampens output volatility by this factor
            'zakat_stabilizer': 0.025,    # Zakat as automatic stabilizer
        }
        
        self.params.update(pakistan_transition_params)
        if params:
            self.params.update(params)
    
    def transition_path(self, scenario='gradual', periods=60):
        """
        Compute the transition path for alpha_t (Islamic share).
        
        Scenarios:
        - 'gradual': 15 years (60 quarters), sigmoid shape
        - 'accelerated': 7 years (28 quarters), steeper sigmoid
        - 'shock': 3 years (12 quarters), near-step function
        """
        p = self.params
        alpha_0 = p['alpha_0']
        alpha_target = p['alpha_target']
        
        t = np.arange(periods)
        
        if scenario == 'gradual':
            # Sigmoid: slow start, acceleration, slow finish
            midpoint = periods * 0.5
            steepness = 0.08
        elif scenario == 'accelerated':
            midpoint = periods * 0.35
            steepness = 0.15
            periods = min(periods, 28)
            t = np.arange(periods)
        elif scenario == 'shock':
            midpoint = periods * 0.2
            steepness = 0.4
            periods = min(periods, 12)
            t = np.arange(periods)
        else:
            raise ValueError(f'Unknown scenario: {scenario}')
        
        sigmoid = 1 / (1 + np.exp(-steepness * (t - midpoint)))
        alpha_t = alpha_0 + (alpha_target - alpha_0) * sigmoid
        
        return alpha_t
    
    def transition_uncertainty(self, alpha_t):
        """
        Uncertainty premium during transition.
        Peaks when alpha is around 0.5 (maximum dual-system complexity),
        then declines as institutions mature.
        """
        p = self.params
        # Hump-shaped: peaks at alpha=0.5
        uncertainty = p['uncertainty_peak'] * 4 * alpha_t * (1 - alpha_t)
        return uncertainty
    
    def transition_cost(self, alpha_t, dalpha_dt):
        """
        GDP cost of transition.
        Proportional to the speed of transition (faster = more costly)
        and inversely proportional to institutional readiness.
        """
        p = self.params
        # Cost proportional to speed of change
        speed_cost = p['transition_cost'] * np.abs(dalpha_dt) * 10
        # Institutional learning reduces cost over time
        return speed_cost
    
    def simulate_transition(self, scenario='gradual', periods=60):
        """
        Simulate macroeconomic outcomes during transition.
        
        Returns dict with time series of key variables.
        """
        p = self.params
        alpha_t = self.transition_path(scenario, periods)
        n = len(alpha_t)
        
        # Initialize output arrays
        output = np.zeros(n)
        inflation = np.zeros(n)
        policy_rate = np.zeros(n)
        uncertainty = np.zeros(n)
        gdp_cost = np.zeros(n)
        pls_benefit = np.zeros(n)
        
        # Random shocks (seed for reproducibility)
        rng = np.random.default_rng(42)
        demand_shocks = rng.normal(0, 0.008, n)
        supply_shocks = rng.normal(0, 0.005, n)
        
        for t in range(n):
            # Transition uncertainty
            uncertainty[t] = self.transition_uncertainty(alpha_t[t])
            
            # Transition speed cost
            dalpha = alpha_t[t] - alpha_t[t-1] if t > 0 else 0
            gdp_cost[t] = self.transition_cost(alpha_t[t], dalpha)
            
            # PLS dampening benefit (increases with Islamic share)
            pls_benefit[t] = p['pls_dampening'] * alpha_t[t]
            
            # Output: base + shocks - uncertainty - transition cost + PLS benefit
            shock_dampening = 1 - pls_benefit[t]
            output[t] = (demand_shocks[t] * shock_dampening 
                        - uncertainty[t] 
                        - gdp_cost[t])
            
            # Inflation: base + supply shocks + uncertainty pass-through
            inflation[t] = (p['pi_bar'] 
                          + supply_shocks[t] 
                          + 0.3 * uncertainty[t])
            
            # Blended policy rate
            conv_rate = p['pi_bar'] + p['phi_pi'] * inflation[t]
            islamic_rate = p['pi_bar'] + p['profit_rate_spread'] + 0.8 * p['phi_pi'] * inflation[t]
            policy_rate[t] = (1 - alpha_t[t]) * conv_rate + alpha_t[t] * islamic_rate
            
            # Autoregressive dynamics
            if t > 0:
                output[t] += 0.7 * output[t-1]
                inflation[t] += 0.5 * (inflation[t-1] - p['pi_bar'])
        
        return {
            'periods': np.arange(n),
            'alpha': alpha_t,
            'output': output,
            'inflation': inflation * 400,  # Annualized
            'policy_rate': policy_rate * 400,  # Annualized
            'uncertainty': uncertainty * 100,  # Percentage points
            'gdp_cost': gdp_cost * 100,  # Percentage points
            'pls_benefit': pls_benefit * 100,  # Percentage points
            'scenario': scenario,
        }

# Instantiate and run all three scenarios
model_transition = PakistanTransitionModel()

sim_gradual = model_transition.simulate_transition('gradual', 60)
sim_accelerated = model_transition.simulate_transition('accelerated', 60)
sim_shock = model_transition.simulate_transition('shock', 60)

print('Transition simulations complete.')
print(f'  Gradual:     {len(sim_gradual["periods"])} quarters ({len(sim_gradual["periods"])//4} years)')
print(f'  Accelerated: {len(sim_accelerated["periods"])} quarters ({len(sim_accelerated["periods"])//4} years)')
print(f'  Shock:       {len(sim_shock["periods"])} quarters ({len(sim_shock["periods"])//4} years)')

In [ ]:
# Plot transition scenarios comparison

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Pakistan Islamic Economic Transition: Three Scenarios', 
             fontsize=14, fontweight='bold', y=1.02)

scenarios = [
    (sim_gradual, 'Gradual (15-year)', COLORS['primary']),
    (sim_accelerated, 'Accelerated (7-year)', COLORS['orange']),
    (sim_shock, 'Shock (3-year)', COLORS['red']),
]

# Row 1: Alpha path, Output, Inflation
for sim, label, color in scenarios:
    quarters = sim['periods']
    years = quarters / 4
    axes[0,0].plot(years, sim['alpha'] * 100, label=label, color=color, linewidth=2)
    axes[0,1].plot(years, sim['output'] * 100, label=label, color=color, linewidth=1.5, alpha=0.8)
    axes[0,2].plot(years, sim['inflation'], label=label, color=color, linewidth=1.5, alpha=0.8)

axes[0,0].set_title('Islamic Economy Share (alpha)', fontweight='bold')
axes[0,0].set_ylabel('%')
axes[0,0].legend(fontsize=8)
axes[0,0].axhline(y=50, color='gray', linestyle=':', alpha=0.3)

axes[0,1].set_title('Output Gap', fontweight='bold')
axes[0,1].set_ylabel('% deviation')
axes[0,1].axhline(y=0, color='gray', linestyle='-', alpha=0.3)

axes[0,2].set_title('Inflation (Annualized)', fontweight='bold')
axes[0,2].set_ylabel('%')

# Row 2: Uncertainty, GDP Cost, Policy Rate
for sim, label, color in scenarios:
    quarters = sim['periods']
    years = quarters / 4
    axes[1,0].plot(years, sim['uncertainty'], label=label, color=color, linewidth=2)
    axes[1,1].plot(years, sim['gdp_cost'], label=label, color=color, linewidth=2)
    axes[1,2].plot(years, sim['policy_rate'], label=label, color=color, linewidth=1.5, alpha=0.8)

axes[1,0].set_title('Transition Uncertainty Premium', fontweight='bold')
axes[1,0].set_ylabel('% points')
axes[1,0].set_xlabel('Years')

axes[1,1].set_title('GDP Cost of Transition', fontweight='bold')
axes[1,1].set_ylabel('% of GDP')
axes[1,1].set_xlabel('Years')

axes[1,2].set_title('Blended Policy/Profit Rate', fontweight='bold')
axes[1,2].set_ylabel('%')
axes[1,2].set_xlabel('Years')

plt.tight_layout()
plt.savefig(os.path.join(fig_dir, 'fig_pakistan_transition_scenarios.png'), 
            dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: fig_pakistan_transition_scenarios.png')

### 4.1 Key Findings from Transition Modeling

1. **The gradual scenario minimizes output loss.** The uncertainty premium peaks lower
   and the institutional learning curve has time to operate.

2. **The shock scenario produces the highest GDP cost.** Rapid forced conversion
   (Iran's approach) creates institutional chaos, market uncertainty, and capital flight.
   Iran's experience confirms this: post-1983, the banking system became functionally
   identical to conventional banking but with Islamic labels.

3. **The accelerated scenario (7 years)** is feasible but requires strong institutional
   pre-investment. The Supreme Court ruling's implied timeline falls in this range.

4. **PLS dampening benefits only materialize after genuine risk-sharing is implemented.**
   If the transition simply relabels products (as Iran did), the dampening benefit is zero.

---
## Part V: Comparative Lessons

### 5.1 Iran (1983): The Full Conversion That Wasn't

Iran enacted the Law for Usury-Free Banking in 1983, mandating complete elimination
of interest from the banking system. The result:

- Banks renamed interest rates as "expected profit rates" or "provisional returns"
- The Central Bank of Iran continued to set rates that functioned identically to interest rates
- Depositors never experienced profit-loss sharing (returns were always positive and fixed)
- Corporate borrowing costs remained essentially unchanged
- No goods or services were exchanged in most "Islamic" contracts

**Lesson for Pakistan:** Forced rapid conversion without institutional infrastructure
produces cosmetic compliance, not genuine transformation.

### 5.2 Malaysia: The Dual System That Works (Partially)

Malaysia has built the most sophisticated dual banking system in the world:

- **Islamic Financial Services Act 2013** provides comprehensive regulation
- Islamic banking holds ~40% of total banking assets
- Bank Negara Malaysia regulates both systems under parallel frameworks
- Shariah Advisory Council provides centralized Shariah governance

**What works:** Strong institutions, clear regulation, gradual development over 40 years.

**What doesn't:** Islamic rates still shadow conventional rates. The system is dual in
regulation but not in monetary policy. Bank Negara sets one Overnight Policy Rate that
both systems follow.

**Lesson for Pakistan:** Malaysia proves that institutional development is possible but
also shows that dual systems tend toward convergence rather than divergence.

---
## Part VI: Institutional Design — What's Missing

The critic asked for "sound institutions." Here is what Pakistan needs that does not
currently exist:

| Institution | Purpose | Exists? | Difficulty |
|------------|---------|---------|------------|
| **Islamic Interbank Benchmark Rate** | Replace KIBOR with a rate derived from actual mudarabah returns | Partially (SBP tracks Islamic interbank rates) | Medium |
| **Central Shariah Board** (independent) | Unified Shariah standard-setting, resolving fragmentation | No (each bank has its own) | Medium |
| **Islamic Deposit Insurance** | Protect depositors in a PLS system where losses are possible | No (current deposit insurance is conventional) | High |
| **Islamic Lender of Last Resort** | Emergency liquidity without interest | Partially (SBP Islamic OMOs) | Medium |
| **Waqf Development Authority** | Manage and grow waqf endowments as economic infrastructure | Exists but dysfunctional | High |
| **Zakat Collection Authority** (reformed) | Effective, transparent, accountable zakat as fiscal tool | Exists but weak collection/distribution | High |
| **Sukuk Rating Methodology** | Separate credit assessment for asset-backed sukuk | Partially (rating agencies cover sukuk) | Low |
| **Islamic Capital Market Infrastructure** | Secondary markets, repo alternatives, hedging tools | Partially | Medium |

### The Governance Challenge

The hardest problem is not financial engineering. It is **governance.** Pakistan's
institutional weakness (corruption, political interference, weak rule of law) affects
conventional and Islamic finance equally. An Islamic economic system requires *more*
institutional strength, not less, because:

1. Profit-loss sharing requires trustworthy accounting and auditing
2. Waqf governance requires protection from political capture
3. Zakat distribution requires transparent, accountable bureaucracy
4. Shariah compliance requires independent, competent scholars

Any transition plan that ignores governance is building on sand.

---
## Part VII: Honest Assessment — What We Don't Know

This section is the most important. A credible analysis must acknowledge its limits.

### Genuinely Unsolved Problems

1. **Monetary policy transmission without interest rates.** No country has demonstrated
   that monetary policy can effectively manage inflation and output stabilization through
   profit-rate targeting alone. This is not a matter of political will. The theoretical
   mechanism for transmitting central bank decisions to the real economy through profit
   rates has not been validated empirically at scale.

2. **Government deficit financing at scale.** Pakistan runs a fiscal deficit of 7%+ of GDP.
   Sukuk can partially replace conventional debt, but scaling to 100% requires either
   (a) sufficient government assets to back the sukuk, or (b) a genuinely new instrument
   that doesn't simply replicate bonds. Neither condition is fully met.

3. **International integration.** Pakistan borrows from the IMF, World Bank, and
   bilateral creditors, all in interest-bearing instruments. A fully Islamic domestic
   system would need to interface with a global financial system that runs on interest.
   There is no proven mechanism for this interface at sovereign scale.

4. **Risk-sharing at macro scale.** While micro-level PLS works (venture capital is
   essentially mudarabah), no economy has demonstrated macro-scale PLS where depositors
   regularly absorb losses. The political and social implications of bank depositors
   losing money in a downturn are untested and potentially destabilizing.

5. **The measurement problem.** A profit-rate-based system requires reliable, real-time
   data on actual business profits across the economy. Pakistan's documentation and tax
   compliance rates are among the lowest in the region, making any profit-based benchmark
   vulnerable to manipulation.

### What This Means

These are not arguments against Islamic economics. They are arguments for **honest,
incremental, institution-building work** rather than ideological declarations. The path
forward is not "abolish interest" but "build the institutions, instruments, and data
infrastructure that would make a genuine alternative possible."

Pakistan is better positioned than most countries for this work: it has a legal mandate
(Supreme Court ruling), growing Islamic banking infrastructure (20%+ and rising), and a
population that supports the goal. What it needs is the institutional patience to build
properly rather than the political pressure to declare victory prematurely.

---
## References

- State Bank of Pakistan, Islamic Banking Department — Quarterly Bulletins
- IMF Article IV Consultation Reports — Pakistan
- Federal Shariat Court of Pakistan — Riba Judgment (2024)
- Bank Negara Malaysia — Islamic Financial Services Act 2013
- Central Bank of Iran — Usury-Free Banking Operations Reports
- IFSB — Islamic Financial Services Industry Stability Report 2023
- Pakistan Ministry of Finance — GoP Asset Light Sukuk Framework
- IMF Working Paper WP/16/72 — Monetary Policy in the Presence of Islamic Banking